# 04 - Explicabilidade com SHAP

Explica as decisoes do modelo final (LightGBM, salvo em `models/lightgbm.joblib`) usando SHAP (SHapley Additive exPlanations) — essencial em credito, onde decisoes automatizadas de negacao/aprovacao precisam ser justificaveis (inclusive por exigencia regulatoria em varios mercados).

Duas visoes complementares:
- **Global**: quais variaveis, em media, mais pesam nas decisoes do modelo.
- **Local**: por que o modelo deu um score especifico para um cliente especifico.

## Setup

In [ ]:
import sys
sys.path.append("..")

import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

from src.train import split_data
from src.evaluate import explain_model


## Carregar dados e modelo final

In [ ]:
df = pd.read_csv("../data/processed/credit_features.csv")
_, X_test, _, y_test = split_data(df)

pipeline = joblib.load("../models/lightgbm.joblib")
print("Modelo carregado:", pipeline.named_steps["classifier"])


## Gerar explicacoes SHAP

Usa uma amostra do conjunto de teste (dados que o modelo nunca viu no treino) para gerar as explicacoes — ver `explain_model()` em `src/evaluate.py`.

In [ ]:
shap_values, X_transformed_df = explain_model(pipeline, X_test, sample_size=1000)
print(f"Explicacoes geradas para {shap_values.shape[0]} clientes e {shap_values.shape[1]} variaveis.")


## Importancia global (beeswarm)

Cada ponto e um cliente. A posicao horizontal mostra o impacto daquela variavel na previsao (positivo = empurra para inadimplencia); a cor mostra se o valor da variavel para aquele cliente e alto (vermelho) ou baixo (azul).

In [ ]:
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.tight_layout()
plt.show()


## Ranking de importancia (media do valor absoluto do SHAP)

In [ ]:
shap.plots.bar(shap_values, max_display=15, show=False)
plt.tight_layout()
plt.show()


**Nota:** espera-se que `EverPastDue`/`TotalTimesPastDue` (atraso previo) e `RevolvingUtilizationOfUnsecuredLines` (utilizacao do limite) fiquem entre as variaveis mais importantes — sao, respectivamente, o sinal mais direto de comportamento de risco e de dependencia de credito rotativo, dois indicadores classicos em modelos de credito. Preencha aqui, apos rodar, qual foi o ranking real observado e se bateu com a expectativa.

## Explicacao local: um cliente de alto risco vs. um de baixo risco

O grafico *waterfall* mostra, para um unico cliente, como cada variavel empurrou o score final para cima ou para baixo a partir do valor medio (base value) do modelo.

In [ ]:
pred_proba = pipeline.named_steps["classifier"].predict_proba(X_transformed_df)[:, 1]

idx_alto_risco = int(np.argmax(pred_proba))
idx_baixo_risco = int(np.argmin(pred_proba))

print(f"Cliente de alto risco -> probabilidade prevista: {pred_proba[idx_alto_risco]:.3f}")
print(f"Cliente de baixo risco -> probabilidade prevista: {pred_proba[idx_baixo_risco]:.3f}")


### Cliente de alto risco

In [ ]:
shap.plots.waterfall(shap_values[idx_alto_risco], show=False)
plt.tight_layout()
plt.show()


### Cliente de baixo risco

In [ ]:
shap.plots.waterfall(shap_values[idx_baixo_risco], show=False)
plt.tight_layout()
plt.show()


## Conclusoes

- Explicacoes SHAP geradas para o modelo final (LightGBM), nas visoes global (beeswarm/bar) e local (waterfall para casos individuais).
- As variaveis mais importantes devem orientar a conversa com a area de negocio sobre quais informacoes do cliente mais pesam na decisao — preencher apos a execucao com o ranking real observado.
- Com modelagem e explicabilidade concluidas, os proximos passos do roadmap sao: indicadores de negocio (Roll Rate, Aging, Taxa de Cura em `src/business_metrics.py`) e o dashboard em Power BI.